# Implement Positional Encoding

![Alt text](../showcase_images/from_paper/pos_encoding.png)

* From Figure 1 in the paper.


$$PE_{(pos, 2i)} = sin(pos/10000^{2 i /d_{model}})$$

$$PE_{(pos, 2i+1)} = cos(pos/10000^{2 i /d_{model}})$$

**Note**:
- In order for the model to make use of the order of the sequence, we inject some information about the relative or absolute position of the tokens in the sequence.
- The position encodings have the same dimension $d_{model}$ as the embeddings
- $pos$ is the position.
- $i$ is the dimension. Each dimension corresponds to a sinusoid.

- **Regularization**:
  - Dropout from paper: "In addition, we apply dropout to the sums of the embeddings and the positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of $p_{drop} = 0.1$.

In [2]:
import torch.nn as nn
import torch
import math

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len = 5000, dropout=0.1):
        # TODO what is the purpose of max_seq_len? could it be named something else?
        """
        # TODO add docstring
        Args:
            d_model: Dimensionality of the vectors.
            max_seq_len:
            dropout: Dropout regularization
        """
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # === Compute the positional encodings once in log space.
        pos_enc = torch.zeros(max_seq_len, d_model)

        # Create a vector of positions [0, 1, ..., max_seq_len-1]
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(-1)

        # Calculate division denominator, 2 correlates to 2i in formula
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10_000.0) / d_model)
        )

        # Fill the pos_enc matrix
        pos_enc[:, 0::2] = torch.sin(position * div_term) # Even indices get sine
        pos_enc[:, 1::2] = torch.cos(position * div_term) # Odd indices get cosine


        pos_enc = pos_enc.unsqueeze(0) # Add a batch -> (1, max_seq_len, d_model)

        # Register as buffer so it is saved with the model, but is not a learned parameter.
        self.register_buffer("pos_enc", pos_enc)
    
    def forward(self, x):
        """x.shape: (batch, seq_len, d_model)"""
        # x = x + self.pos_enc[:, : x.size(1)].requires_grad_(False)
        x = x + self.pos_enc[:,  :x.size(1), :]
        return self.dropout(x)

In [4]:
def test():
    print(f"\n\nTesting Positional Encoding...")
    pos_enc_layer = PositionalEncoding(d_model=512, max_seq_len=100)
    sample_input = torch.zeros(1, 100, 512)
    output = pos_enc_layer(sample_input)

    print(output.shape)

test()



Testing Positional Encoding...
torch.Size([1, 100, 512])
